# Email Automation

In [2]:
import pandas as pd
import smtplib
import os
import re
from email.mime.text import MIMEText

# ========= CONFIG =========
EMAIL = "abubakarp496@gmail.com"
PASSWORD = "i hide this password"   # app password 
EXCEL_PATH = r"C:\Users\Abubakar\OneDrive\Desktop\vendor_data.xlsx"

# ========= EMAIL VALIDATION =========
def is_valid_email(email):
    return re.match(r"^[\w\.-]+@[\w\.-]+\.\w+$", str(email))

# ========= LOAD FILE =========
if not os.path.exists(EXCEL_PATH):
    print("❌ File not found")
    raise SystemExit

try:
    df = pd.read_excel(EXCEL_PATH)
except Exception:
    print("❌ Close Excel file OR move it out of OneDrive")
    raise SystemExit

# normalize columns
df.columns = df.columns.str.strip().str.lower()

# ========= SMTP =========
try:
    server = smtplib.SMTP("smtp.gmail.com", 587)
    server.starttls()
    server.login(EMAIL, PASSWORD)
except Exception as e:
    print("❌ Email login failed:", e)
    raise SystemExit

# ========= LOOP =========
for _, row in df.iterrows():
    try:
        vendor = str(row.get("vendor_name", "")).strip()
        email = str(row.get("email", "")).strip()
        item = str(row.get("item", "")).strip()
        amount = str(row.get("amount", "")).strip()
        status = str(row.get("payment_status", "")).strip().lower()

        if not is_valid_email(email):
            continue

        # ========= CONDITIONS =========
        if status == "overdue":
            subject = f"Overdue Payment - {item}"
            body = f"""Dear {vendor},

Your payment of {amount} is overdue.

Regards,
Accounts Team"""

        elif status == "pending":
            subject = f"Payment Reminder - {item}"
            body = f"""Dear {vendor},

This is a reminder for your pending payment of {amount}.

Regards,
Accounts Team"""

        elif status == "paid":
            subject = f"Payment Received - {item}"
            body = f"""Dear {vendor},

We have received your payment. Thank you.

Regards,
Accounts Team"""

        elif status == "not required":
            subject = f"No Payment Required - {item}"
            body = f"""Dear {vendor},

No payment is required.

Regards,
Accounts Team"""

        else:
            subject = f"Invoice Update - {item}"
            body = f"""Dear {vendor},

Please review your invoice.

Regards,
Accounts Team"""

        msg = MIMEText(body)
        msg["Subject"] = subject
        msg["From"] = EMAIL
        msg["To"] = email

        server.sendmail(EMAIL, email, msg.as_string())

    except Exception:
        continue

server.quit()
print("✅ All emails are sent")

✅ All emails are sent
